# Complete missing registration timings
Rerun only evaluation stages without a recorded fresh runtime. Existing measured PASTE2 and MLP times are retained. This requires the original datasets and environment, but does not execute another notebook. Publication scores and registered coordinates are never replaced.

This is an expensive fresh-fit pass, not cache-load timing. Completed timing jobs survive restarts; an interrupted fit starts afresh. CAST and Splat family totals include their affine and non-affine variants. The rigid splat pose is timed separately.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from dotenv import load_dotenv
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').exists() and (p/'paper').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT/'.env')
from paper.workflows import registration_timing_completion as timing
pd.options.display.float_format = '{:.4g}'.format

# Set an explicit exported bundle directory to override automatic selection.
BUNDLE_PATH = None
if BUNDLE_PATH is None:
    manifests = sorted((ROOT/'notebooks/checkpoints/registration_publication_controls').glob('*/reports/plot_data/*/manifest.json'), key=lambda p: p.parent.name, reverse=True)
    complete = [p.parent for p in manifests if json.loads(p.read_text()).get('complete')]
    if not complete:
        raise FileNotFoundError('Export plotting data from the completed publication experiment first.')
    BUNDLE_PATH = complete[0]
BUNDLE_PATH = Path(BUNDLE_PATH).resolve()
print('Source bundle:', BUNDLE_PATH)
manifest, tables, TIMING_OUTPUT, timing_log = timing.prepare(ROOT, BUNDLE_PATH)
print('Isolated timing workspace:', TIMING_OUTPUT)


Source bundle: /home/javeda/Documents/Research/spatsplat-registration/notebooks/checkpoints/registration_publication_controls/5b2016112ab99603ff0c/reports/plot_data/20261101T214518078609Z


/home/javeda/Documents/Research/spatsplat-registration/paper/workflows/registration_plot_data.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, float_precision='round_trip', keep_default_na=False, na_values=[''])


Isolated timing workspace: /home/javeda/Documents/Research/spatsplat-registration/notebooks/checkpoints/registration_timing_completion/9aab12360df066232562


In [2]:
plan = timing.missing_plan(tables, TIMING_OUTPUT)
print('Missing evaluation stages:', len(plan))
display(plan.groupby(['dataset', 'stage']).size().rename('fresh_fits').to_frame())
display(plan[plan.stage == 'PASTE2 full pipeline'])
print('Any PASTE2 cases listed above require a full fresh fit, including overlap selection; allow substantial time.')


Missing evaluation stages: 126


fresh_fits
dataset stage                           
Allen   CAST family                    4
        Geometry family                4
        PASTE family                   4
        PASTE2 full pipeline           1
        Splat family                   4
        Splat rigid pose               4
BARseq  CAST family                   21
        Geometry family               21
        PASTE family                  21
        Splat family                  21
        Splat rigid pose              21

,case,dataset,stage
5,allen_28_29_natural,Allen,PASTE2 full pipeline


Any PASTE2 cases listed above require a full fresh fit, including overlap selection; allow substantial time.


In [3]:
cases = timing.load_evaluation_cases(manifest['protocol'], tables) if not plan.empty else []
print('Evaluation cases loaded:', len(cases))


Evaluation cases loaded: 25


In [4]:
RUN_MISSING_TIMINGS = True
if RUN_MISSING_TIMINGS:
    remaining = timing.run_missing(cases, manifest['protocol'], tables, ROOT, TIMING_OUTPUT, timing_log)
else:
    remaining = timing.missing_plan(tables, TIMING_OUTPUT)
print('Remaining stages:', len(remaining))
display(remaining)
if (TIMING_OUTPUT/'failures.csv').exists():
    display(pd.read_csv(TIMING_OUTPUT/'failures.csv'))


allen_28_29_natural missing: CAST family, Geometry family, PASTE family, PASTE2 full pipeline, Splat family, Splat rigid pose
Fitting Allen Geometry: b930505ca6ae78b2a92c
Fitting Allen PASTE: 294fc30977343f9072ba
Using selected backend cpu. If you want to use gpu, set use_gpu = True.
Fitting Allen CAST: 2bdbc7a0ecc4c4584ce8
Constructing delaunay graphs for 2 samples...


/home/javeda/miniconda3/envs/spatsplat/lib/python3.11/site-packages/CAST/utils.py:15: FutureWarning: The 'convex hull' option for the 'clip' parameter is deprecated and will be removed in a future release. Use 'convex_hull' instead.
  cells, _ = voronoi_frames(coords, clip="convex hull")
/home/javeda/miniconda3/envs/spatsplat/lib/python3.11/site-packages/CAST/utils.py:15: FutureWarning: The 'as_gdf' parameter currently defaults to True but will default to False in a future release. Set it explicitly to avoid this warning.
  cells, _ = voronoi_frames(coords, clip="convex hull")
/home/javeda/miniconda3/envs/spatsplat/lib/python3.11/site-packages/CAST/utils.py:15: FutureWarning: The 'return_input' parameter currently defaults to True but will default to False in a future release. Set it explicitly to avoid this warning.
  cells, _ = voronoi_frames(coords, clip="convex hull")
/home/javeda/miniconda3/envs/spatsplat/lib/python3.11/site-packages/CAST/utils.py:16: FutureWarning: `use_index` de

Training on cpu...


Loss: -407.994 step time=0.361s: 100%|██████████| 400/400 [02:08<00:00,  3.11it/s]


Finished.
The embedding, log, model files were saved to /home/javeda/Documents/Research/spatsplat-registration/notebooks/checkpoints/registration_timing_completion/9aab12360df066232562/attempts/allen_28_29_natural/CAST/9a038680fa2047afa52999b171ee796e/notebooks/checkpoints/registration_standardized/fits/CAST/2bdbc7a0ecc4c4584ce8/work


Loss: 419.748: 100%|██████████| 1500/1500 [00:26<00:00, 56.58it/s]
/home/javeda/miniconda3/envs/spatsplat/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loss: 145.901: 100%|██████████| 800/800 [00:22<00:00, 36.07it/s]


... (155 outputs left out)

Loss: 604.661: 100%|██████████| 800/800 [00:10<00:00, 78.57it/s]


Fitting BARseq Splat: 125864c8761ffabf0d01
AE epoch 1: train=4.67, validation cosine=0.4052
AE epoch 25: train=4.074, validation cosine=0.636
AE epoch 50: train=3.899, validation cosine=0.7232
AE epoch 75: train=3.885, validation cosine=0.7424
AE epoch 100: train=3.847, validation cosine=0.7613
AE epoch 125: train=3.779, validation cosine=0.7869
AE epoch 150: train=3.759, validation cosine=0.7918
Section 0 | training budget: {'training_measurements': 1447, 'updates_per_epoch': 3, 'epochs': 500, 'planned_optimizer_steps': 1500, 'mae_refine_start_epoch': 375}
Section 0 | epoch 0: {'optimizer_steps': 3, 'train': {'joint_loss': 0.3658, 'cell_loss': 0.0, 'composition_loss': 0.271, 'total_count_loss': 0.3576, 'off_tissue_loss': 2.328, 'center_support_loss': 0.001117}, 'val': {'loss': 0.4963, 'mae': 0.4963, 'n': 1447}, 'phase': 'composition'}
Section 0 | epoch 24: {'optimizer_steps': 75, 'train': {'joint_loss': 0.338, 'cell_loss': 0.0, 'composition_loss': 0.2602, 'total_count_loss': 0.3569, '

,case,dataset,stage


,case,stage,error


## Export updated plotting data
The new snapshot copies scores, availability, measurements, and registered maps unchanged. Only timing values and timing provenance are added. Failed or unfinished stages remain explicitly unknown. Open `registration_publication_plots-allen_barseq.ipynb` and rerun its loading and runtime-figure cells; automatic selection uses the newest complete bundle, or set its `BUNDLE_PATH` to the printed path.

In [5]:
NEW_BUNDLE = timing.publish_timings(BUNDLE_PATH, tables, TIMING_OUTPUT)
print('Updated plotting bundle:', NEW_BUNDLE)
remaining = timing.missing_plan(tables, TIMING_OUTPUT)
print('Evaluation timing complete:', remaining.empty)
if not remaining.empty:
    display(remaining)


Updated plotting bundle: /home/javeda/Documents/Research/spatsplat-registration/notebooks/checkpoints/registration_publication_controls/5b2016112ab99603ff0c/reports/plot_data/20261102T025630560124Z
Evaluation timing complete: True
